# 03 — YOLO Litter Detection: Baseline (Task 2)

**Project:** CNN-Based Aerial Litter Detection for Sustainable Trail and Environmental Cleanup  
**Module:** ST7088CEM Artificial Neural Networks

Task 2, baseline: a pretrained Ultralytics YOLO detector is fine-tuned on
UAVVaste (single class `rubbish`) at **standard 640 px** settings. This
establishes the reference point that the optimised configuration (notebook 04 —
higher resolution, augmentation, SAHI) is measured against.
Metrics: mAP@0.5, mAP@0.5:0.95, precision, recall.

## 1. Environment setup (Kaggle GPU + Internet)

Enable **GPU** and **Internet** in the notebook settings (Internet is needed to
`pip install ultralytics` and to download the pretrained weights).

In [ ]:
# On Kaggle (Internet enabled), uncomment:
# !git clone -b feature/yolo-baseline https://github.com/Sajan491/STW7088CEM-ANN-Assignment.git
# %cd STW7088CEM-ANN-Assignment
# import os; os.makedirs('data', exist_ok=True)
# !ls /kaggle/input
# !ln -s /kaggle/input/<dataset-slug>/data/images data/images
# !ln -s /kaggle/input/<dataset-slug>/data/annotations data/annotations
%pip install -q ultralytics

import os, sys, platform
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')
sys.path.insert(0, str(Path.cwd()))

import torch, ultralytics
print('Working directory:', Path.cwd())
print('Machine:', platform.node(), '|', platform.platform())
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('Ultralytics:', ultralytics.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Build the YOLO dataset

Reuses the leakage-free image-level splits from Phase 1 and converts COCO
annotations to YOLO format. Images carrying EXIF rotation tags (Ultralytics
would otherwise auto-rotate them and misalign the boxes) are written in raw
pixel space, so labels and pixels share one coordinate system.

In [ ]:
if not Path('data/processed/splits.json').exists():
    !python -m src.data.splits
!python -m src.data.coco_to_yolo

## 3. Fine-tune the baseline detector

Configuration in `configs/yolo_baseline.yaml` (model, 640 px, epochs, batch,
early-stopping patience). Ultralytics training plots and the best checkpoint
are copied into `results/figures/` and `checkpoints/`.

In [ ]:
!python -m src.training.train_yolo

## 4. Evaluate on the held-out test split

In [ ]:
!python -m src.evaluation.evaluate_yolo

## 5. Results

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

print('Test-split detection metrics:')
display(pd.read_csv('results/tables/yolo_baseline_metrics.csv'))
for f in ['yolo_baseline_results.png', 'yolo_baseline_confusion_matrix.png',
          'yolo_baseline_pr_curve.png']:
    p = Path('results/figures') / f
    if p.exists():
        display(IPImage(str(p), width=760))

## 6. Notes for the report

- This is the **baseline** at standard 640 px input. Because ~99% of litter
  objects occupy under 1% of the frame (see notebook 01), downscaling large
  aerial images to 640 px shrinks most objects to a handful of pixels — so a
  modest baseline mAP is expected, and is the motivation for the optimised
  configuration in notebook 04.
- Splits are the same image-level splits used throughout, so detection and the
  tile classifier are evaluated on comparable held-out images.
- The baseline checkpoint (`checkpoints/yolo_baseline.pt`) is reused for the
  qualitative demonstration in notebook 05.